# Store Sales — forecasting 1,782 series sixteen days ahead

Daily sales for 54 Corporación Favorita stores × 33 product families in
Ecuador, forecast sixteen days out and scored on RMSLE.

**What this notebook is honest about.** It reaches **0.40695** on the public
leaderboard, rank ~92 of 642. The top is 0.37294 and the top 5% is 0.38369, so
the gap is 0.023 and this is not a winning solution. What it does have is a
measurement protocol that says *why* — including one result that surprised me:
two substantially different models, 0.021 apart on the fold immediately before
the test window, scored 0.40713 and 0.40695 on the leaderboard. Local gains are
not reaching it.

The reading behind this: [Ekrem Bayar's comprehensive
guide](https://www.kaggle.com/code/ekrembayar/store-sales-ts-forecasting-a-comprehensive-guide),
the most-voted solution notebook on this competition, is where the
zero-forecasting rule comes from; [Ryan Holbrook's Time Series
course](https://www.kaggle.com/learn/time-series) is where the
deterministic-plus-learned framing comes from.

Code, tests and the full write-up:
**[github.com/conchocon154/store-sales-forecasting](https://github.com/conchocon154/store-sales-forecasting)**

## Three decisions that set the result

**Work in log space, throughout.** RMSLE is RMSE on `log1p`, so a model fitted
on raw sales optimises a loss the competition does not use. On series running
from 0 to 124,717 that is not a rounding difference. Every model here — the
baselines' averages included — is computed on `log1p` and converted back once,
at the end.

**Forecast directly, not recursively.** On the day the forecast is made the
last known sale is yesterday, and day sixteen still has to be predicted. A
one-step model that feeds its own output back in compounds its error sixteen
times. Here each training row is *(series, origin, horizon)*: the per-series
history is summarised strictly before the origin, the calendar features describe
the target day, and "sixteen days out" is learned as its own problem.

**Answer the dead series with a rule.** Ninety-six of the 1,782 series sold
nothing at all in the last sixty days, and they cover 1,536 of the 28,512 rows
to predict. Their forecast is zero — that is the answer, not an estimate — so
they are dropped from the fit and filled in afterwards.

In [ ]:
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

DATA = Path("/kaggle/input/store-sales-time-series-forecasting")
pd.set_option("display.width", 120)

## The metric, and the folds it has to be measured on

In [ ]:
from __future__ import annotations

"""The metric, and the only honest way to measure against it here.

RMSLE is RMSE on `log1p`, so every model in this repo predicts in log space and
is only converted back at the end. That is not a convenience: fitting on raw
sales optimises the wrong loss, and on a series whose values run from 0 to
124,717 the difference is not small.
"""



HORIZON = 16          # the test period, 2017-08-16 to 2017-08-31


def rmsle(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0.0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))


def rmse_log(y_true_log, y_pred_log) -> float:
    """The same number, when both sides are already in log space."""
    d = np.asarray(y_pred_log, float) - np.asarray(y_true_log, float)
    return float(np.sqrt(np.mean(d ** 2)))


def make_folds(last_train_date: pd.Timestamp, n: int = 3,
          horizon: int = HORIZON) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    """Backtest origins, each holding out the next `horizon` days.

    The folds are the same length as the competition's test window and sit
    directly before it, so a fold answers the question the leaderboard asks —
    "sixteen days ahead, from a standing start" — rather than the much easier
    one-day-ahead question a random split would ask.
    """
    out = []
    end = pd.Timestamp(last_train_date)
    for _ in range(n):
        start = end - pd.Timedelta(days=horizon - 1)
        out.append((start, end))
        end = start - pd.Timedelta(days=1)
    return list(reversed(out))

The folds are sixteen-day blocks ending at the last day of training, stepping
backwards without overlap:

```
… train ……………………│ 06-29 → 07-14 │ 07-15 → 07-30 │ 07-31 → 08-15 │  test: 08-16 → 08-31
```

A random train/test split would ask a much easier question — predict a Tuesday
given the Wednesday after it — and would rank these models differently.

## Loading and joining the seven files

In [ ]:
from __future__ import annotations

"""Loading and joining the seven files the competition ships.

The panel is dense on purpose: 54 stores x 33 families x every date, with the
gaps filled at zero. Every calendar-aware model below assumes a row exists for
each series on each day, and the raw train file does not guarantee that.
"""





# Ecuador's public sector is paid on the 15th and the last day of the month,
# and the competition's own data description calls this out as a driver.
PAYDAYS = (15,)

# The Manabi earthquake, 16 April 2016. Relief buying distorts the weeks after
# it, so the model gets to see the window rather than learning it as noise.
QUAKE = pd.Timestamp("2016-04-16")
QUAKE_WEEKS = 8


@dataclass(frozen=True)
class Frames:
    train: pd.DataFrame
    test: pd.DataFrame
    stores: pd.DataFrame
    oil: pd.DataFrame
    holidays: pd.DataFrame
    transactions: pd.DataFrame


def load(data_dir: Path | str = DATA) -> Frames:
    d = Path(data_dir)
    dates = ["date"]
    return Frames(
        train=pd.read_csv(d / "train.csv", parse_dates=dates),
        test=pd.read_csv(d / "test.csv", parse_dates=dates),
        stores=pd.read_csv(d / "stores.csv"),
        oil=pd.read_csv(d / "oil.csv", parse_dates=dates),
        holidays=pd.read_csv(d / "holidays_events.csv", parse_dates=dates),
        transactions=pd.read_csv(d / "transactions.csv", parse_dates=dates),
    )


def build_panel(frames: Frames) -> pd.DataFrame:
    """One row per (store, family, date) across train and test, sales NaN on test."""
    train = frames.train.assign(split="train")
    test = frames.test.assign(sales=np.nan, split="test")
    cols = ["id", "date", "store_nbr", "family", "sales", "onpromotion", "split"]
    panel = pd.concat([train[cols], test[cols]], ignore_index=True)

    full = _dense_index(panel)
    panel = full.merge(panel, on=["date", "store_nbr", "family"], how="left")
    panel["onpromotion"] = panel["onpromotion"].fillna(0.0)
    panel["split"] = panel["split"].fillna("train")
    panel.loc[panel["split"] == "train", "sales"] = \
        panel.loc[panel["split"] == "train", "sales"].fillna(0.0)

    panel = panel.merge(frames.stores, on="store_nbr", how="left")
    return panel.sort_values(["store_nbr", "family", "date"], ignore_index=True)


def _dense_index(panel: pd.DataFrame) -> pd.DataFrame:
    dates = pd.date_range(panel["date"].min(), panel["date"].max(), freq="D")
    stores = np.sort(panel["store_nbr"].unique())
    families = np.sort(panel["family"].unique())
    idx = pd.MultiIndex.from_product([dates, stores, families],
                                     names=["date", "store_nbr", "family"])
    return idx.to_frame(index=False)


def oil_series(oil: pd.DataFrame, index: pd.DatetimeIndex) -> pd.Series:
    """Daily WTI, forward-filled.

    The file has weekday quotes with gaps, and the first row is empty, so it is
    reindexed onto every calendar day and filled in both directions — a missing
    price is a market holiday, not a change in price.
    """
    s = (oil.set_index("date")["dcoilwtico"]
            .reindex(index)
            .ffill()
            .bfill())
    s.name = "oil"
    return s


def holiday_flags(holidays: pd.DataFrame, stores: pd.DataFrame,
                  index: pd.DatetimeIndex) -> pd.DataFrame:
    """A per-(date, store) view of whether the day is off.

    Three things the raw file makes easy to get wrong, all of them documented
    in the competition's data description:

      * `transferred` is True on the *original* date, which was worked, and a
        separate row of type `Transfer` carries the day people actually took.
      * `Work Day` is the opposite of a holiday: a Saturday worked to make up
        for a bridge.
      * `locale` scopes the holiday to the country, a region or a single city,
        so a Local holiday only closes the stores in that city.
    """
    h = holidays.copy()
    h = h[h["type"] != "Work Day"]
    h = h[~h["transferred"].astype(bool)]
    h = h[h["type"] != "Event"]

    national = set(h.loc[h["locale"] == "National", "date"])
    regional = set(zip(h.loc[h["locale"] == "Regional", "date"],
                       h.loc[h["locale"] == "Regional", "locale_name"]))
    local = set(zip(h.loc[h["locale"] == "Local", "date"],
                    h.loc[h["locale"] == "Local", "locale_name"]))
    workdays = set(holidays.loc[holidays["type"] == "Work Day", "date"])

    rows = []
    for _, store in stores.iterrows():
        flags = pd.DataFrame({"date": index})
        flags["store_nbr"] = store["store_nbr"]
        flags["is_national_holiday"] = flags["date"].isin(national).astype("int8")
        flags["is_regional_holiday"] = [
            (d, store["state"]) in regional for d in index]
        flags["is_local_holiday"] = [
            (d, store["city"]) in local for d in index]
        flags["is_work_day"] = flags["date"].isin(workdays).astype("int8")
        rows.append(flags)

    out = pd.concat(rows, ignore_index=True)
    for c in ("is_regional_holiday", "is_local_holiday"):
        out[c] = out[c].astype("int8")
    out["is_holiday"] = ((out[["is_national_holiday", "is_regional_holiday",
                               "is_local_holiday"]].max(axis=1) == 1)
                         & (out["is_work_day"] == 0)).astype("int8")
    return out

Three things the holiday file makes easy to get wrong, all of them documented in
the competition's own data description:

* `transferred` is `True` on the day that was **worked** — a separate `Transfer`
  row carries the day people actually took off;
* `Work Day` is the *opposite* of a holiday: a Saturday worked to make up for a
  bridge;
* `locale` scopes a holiday to the country, a region or one city, so a Local
  holiday closes only the stores in that city.

Getting any of the three backwards is invisible in the output, which is why the
repo has a test for each.

In [ ]:
frames = load(DATA)
panel = build_panel(frames)
train = panel[panel["split"] == "train"]
test = panel[panel["split"] == "test"]

print("train ", train.date.min().date(), "→", train.date.max().date())
print("series", train.groupby(["store_nbr", "family"]).ngroups,
      "  test days", test.date.nunique(), "  test rows", len(test))
print("zero rows in train: %.1f%%" % (100 * (train.sales == 0).mean()))

# The dead-series fact, computed here rather than with the helper below,
# because it is the observation that motivates the rule.
origin = test.date.min()
last60 = train[train.date > origin - pd.Timedelta(days=60)]
gone = last60.groupby(["store_nbr", "family"]).sales.sum().pipe(lambda s: s[s == 0].index)
covered = test.set_index(["store_nbr", "family"]).index.isin(set(gone))
print(f"sold nothing in the last 60 days: {len(gone)} series, covering "
      f"{covered.sum()} of {covered.size} rows to predict")

## Features

In [ ]:
from __future__ import annotations

"""Features for a direct sixteen-day-ahead forecast.

The horizon is what shapes this file. On the day the forecast is made, the last
known sale is the day before; the model must still predict day sixteen. So no
feature may depend on anything after the origin, and a lag of one day is
unavailable for fifteen of the sixteen targets.

The way out is a *direct* model: one row per (series, origin, horizon), where
the series statistics are computed strictly up to the origin and the calendar
features describe the target day. Recursive one-step models avoid the lag
problem by feeding predictions back in, which compounds error across sixteen
steps; this does not.
"""




# Windows for the per-series history summaries, in days before the origin.
WINDOWS = (7, 14, 28, 56, 112)
# How many same-weekday observations to average.
DOW_WEEKS = (4, 8)


def calendar(dates: pd.Series) -> pd.DataFrame:
    d = pd.to_datetime(dates)
    out = pd.DataFrame({
        "dayofweek": d.dt.dayofweek.astype("int8"),
        "day": d.dt.day.astype("int8"),
        "month": d.dt.month.astype("int8"),
        "year": d.dt.year.astype("int16"),
        "dayofyear": d.dt.dayofyear.astype("int16"),
        "weekofyear": d.dt.isocalendar().week.astype("int16").to_numpy(),
    })
    out["is_weekend"] = (out["dayofweek"] >= 5).astype("int8")
    # Wages land on the 15th and the last day of the month; the shops feel it.
    out["is_payday"] = (d.dt.day.isin(PAYDAYS) | d.dt.is_month_end).astype("int8")
    out["days_from_payday"] = _days_from_payday(d).astype("int8")
    weeks_since_quake = (d - QUAKE).dt.days / 7.0
    out["quake_window"] = ((weeks_since_quake >= 0) &
                           (weeks_since_quake <= QUAKE_WEEKS)).astype("int8")
    # A smooth trend beats a raw date for a tree: it can split it, and it does
    # not explode when the test period sits past every value it has seen.
    out["t"] = (d - pd.Timestamp("2013-01-01")).dt.days.astype("int32")
    return out


def _days_from_payday(d: pd.Series) -> pd.Series:
    day = d.dt.day
    month_end = d.dt.days_in_month
    to_15 = (15 - day).abs()
    to_end = (month_end - day).abs()
    return np.minimum(to_15, to_end)


# Days before the origin to read off directly. The first week gives the recent
# shape, 14/21/28 give the weekly rhythm, and 364 is the same weekday a year ago
# — which is how a tree gets at an annual season from four years of data.
LAGS = (1, 2, 3, 4, 5, 6, 7, 14, 21, 28, 35, 364)

# The longest window anything below looks back over. Slicing the past to this
# once, instead of grouping over four years for every origin, is what makes a
# year of training origins affordable.
MAX_LOOKBACK = 365


def series_history(panel: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    """Per-series summaries of everything known strictly before `origin`.

    Computed in log space, because that is the space the model is scored in:
    the mean of log1p sales is a far better predictor of log1p sales than the
    log of the mean, on a series that is zero a third of the time.
    """
    past = panel[panel["date"] < origin]
    if past.empty:
        raise ValueError(f"no history before {origin!r}")
    past = past[past["date"] > origin - pd.Timedelta(days=MAX_LOOKBACK + 1)]

    past = past.assign(y=np.log1p(past["sales"].to_numpy(dtype=float)))
    past = past.assign(lag=(origin - past["date"]).dt.days)
    keys = ["store_nbr", "family"]
    out = past.groupby(keys, observed=True)[["y"]].mean().rename(
        columns={"y": "mean_365"})

    last = past["date"].max()
    for w in WINDOWS:
        window = past[past["date"] > last - pd.Timedelta(days=w)]
        agg = window.groupby(keys, observed=True)["y"].agg(["mean", "std", "max"])
        agg.columns = [f"mean_{w}", f"std_{w}", f"max_{w}"]
        out = out.join(agg)
        # Share of days with no sale: separates "quiet" from "dead".
        zero = window.assign(z=(window["sales"] == 0).astype(float))
        out[f"zero_rate_{w}"] = zero.groupby(keys, observed=True)["z"].mean()

    # Same weekday as the target, which is where most of the signal lives.
    for weeks in DOW_WEEKS:
        window = past[past["date"] > last - pd.Timedelta(days=7 * weeks)]
        dow = (window.assign(dayofweek=window["date"].dt.dayofweek)
                     .groupby(keys + ["dayofweek"], observed=True)["y"].mean()
                     .rename(f"dow_mean_{weeks}w"))
        out = out.join(dow.reset_index().pivot_table(
            index=keys, columns="dayofweek", values=f"dow_mean_{weeks}w"
        ).rename(columns=lambda c: f"dow{c}_mean_{weeks}w"))

    # Explicit lags, not just window means. A mean says what the level is; the
    # last seven days in order say what shape it is in — a series that sold
    # 0,0,0,40,40,40,40 and one that sold 20 every day have the same mean and
    # very different futures.
    lag_frame = past.pivot_table(index=keys, columns="lag", values="y",
                                 observed=True)
    for k in LAGS:
        out[f"lag_{k}"] = lag_frame[k] if k in lag_frame.columns else np.nan

    out["promo_mean_28"] = (
        past[past["date"] > last - pd.Timedelta(days=28)]
        .groupby(keys, observed=True)["onpromotion"].mean())

    # Recent slope: the last four weeks against the four before them.
    recent = past[past["date"] > last - pd.Timedelta(days=28)]
    prior = past[(past["date"] <= last - pd.Timedelta(days=28)) &
                 (past["date"] > last - pd.Timedelta(days=56))]
    out["trend_28"] = (recent.groupby(keys, observed=True)["y"].mean()
                       - prior.groupby(keys, observed=True)["y"].mean())

    # Cross-sectional context. A tree cannot derive "this family is having a
    # good month everywhere" or "this store is quiet" from the series' own
    # columns, and both carry signal a single series is too noisy to show.
    for w in (28, 112):
        window = past[past["date"] > last - pd.Timedelta(days=w)]
        fam = window.groupby("family", observed=True)["y"].mean()
        store = window.groupby("store_nbr", observed=True)["y"].mean()
        out[f"fam_mean_{w}"] = out.index.get_level_values("family").map(fam)
        out[f"store_mean_{w}"] = out.index.get_level_values("store_nbr").map(store)
        # Where this series sits against both, which is what actually matters.
        out[f"rel_fam_{w}"] = out[f"mean_{28 if w == 28 else 112}"] - out[f"fam_mean_{w}"]
        out[f"rel_store_{w}"] = out[f"mean_{28 if w == 28 else 112}"] - out[f"store_mean_{w}"]

    return out.fillna(0.0).reset_index()


def first_sale(panel: pd.DataFrame) -> pd.Series:
    """The day each series first sold anything.

    Many series are all zero for months at the start — the family had not been
    stocked yet. Those rows are not a forecasting problem, they are a fact
    about the store's range, and training on them teaches the model to predict
    zero for a series that is now perfectly healthy.
    """
    sold = panel[panel["sales"] > 0]
    return sold.groupby(["store_nbr", "family"], observed=True)["date"].min()


def dead_series(panel: pd.DataFrame, origin: pd.Timestamp,
                lookback: int = 60) -> set[tuple[int, str]]:
    """Series that sold nothing at all in the `lookback` days before the origin.

    Ninety-six of the 1,782 series are in this state at the end of training,
    and they cover 1,536 of the 28,512 rows to be predicted. Forecasting them
    at zero is not a modelling choice, it is the answer — and it keeps a
    gradient booster from spending capacity on rows whose truth is a constant.
    """
    past = panel[(panel["date"] < origin) &
                 (panel["date"] >= origin - pd.Timedelta(days=lookback))]
    total = past.groupby(["store_nbr", "family"], observed=True)["sales"].sum()
    return set(total[total == 0].index)


# A fortnight centred on the same date a year ago. One day 364 days back is a
# single noisy observation; fifteen of them are a season.
YEAR_LAG = 364
YEAR_WINDOW = 15


def seasonal_index(panel: pd.DataFrame) -> pd.DataFrame:
    """How far above its own baseline each series ran at this point last year.

    This is the feature the error decomposition asked for. SCHOOL AND OFFICE
    SUPPLIES is 13% of all squared error on the August fold, at RMSLE 0.873,
    because Ecuadorean term starts and the family goes up several-fold for a
    few weeks. A model that only sees day-of-year has four Augusts to learn
    that from, against thirty-two other families pulling the gradient the other
    way. A column that says "this series ran 1.9 above its usual level on this
    date last year" hands it over directly.

    Returned indexed by the date the value is *for*, so a row for 2017-08-16
    carries what happened around 2016-08-17. Everything it reads is more than a
    year old, so it is available for every forecast day.
    """
    keys = ["store_nbr", "family"]
    df = panel[["date", "store_nbr", "family", "sales"]].copy()
    df["y"] = np.log1p(df["sales"].to_numpy(dtype=float))
    df = df.sort_values(keys + ["date"])

    g = df.groupby(keys, observed=True)["y"]
    df["season_window"] = g.transform(
        lambda v: v.rolling(YEAR_WINDOW, center=True, min_periods=3).mean())
    # The series' own level over the surrounding year, so the index is a shape
    # and not a level — the level is already in the history columns.
    df["season_base"] = g.transform(
        lambda v: v.rolling(365, center=True, min_periods=60).mean())
    df["season_index"] = df["season_window"] - df["season_base"]

    # The same thing pooled across stores. One store-family is thin; a family
    # across fifty-four stores is not.
    fam = (df.groupby(["family", "date"], observed=True)["season_index"]
             .mean().rename("fam_season_index").reset_index())

    out = df[keys + ["date", "season_index"]].merge(
        fam, on=["family", "date"], how="left")
    # Shift forward a year: the row for date D carries what happened at D-364.
    out["date"] = out["date"] + pd.Timedelta(days=YEAR_LAG)
    return out.dropna(subset=["season_index"])

## The model, and the ladder it has to beat

In [ ]:
from __future__ import annotations

"""The model ladder, from the answer you get for free to the one worth fitting.

Every model predicts in log space and returns sales in the original units, and
every one of them is measured on the same backtest, so the table in the README
is a like-for-like comparison rather than a set of numbers from different
protocols.
"""




KEYS = ["store_nbr", "family"]


def _days_to_national(holidays: pd.DataFrame, index: pd.DatetimeIndex) -> pd.Series:
    """Days to the nearest national day off, in either direction.

    Shops empty out the day after a holiday and fill up the day before it, and
    a flag on the day itself cannot say either.
    """
    h = holidays[(holidays["locale"] == "National") &
                 (holidays["type"].isin(["Holiday", "Transfer", "Bridge",
                                         "Additional"])) &
                 (~holidays["transferred"].astype(bool))]
    days = np.sort(pd.DatetimeIndex(h["date"].unique())
                     .to_numpy(dtype="datetime64[D]").astype(np.int64))
    if len(days) == 0:
        return pd.Series(99, index=index, dtype="int16")
    target = index.to_numpy(dtype="datetime64[D]").astype(np.int64)
    pos = np.searchsorted(days, target)
    before = days[np.clip(pos - 1, 0, len(days) - 1)]
    after = days[np.clip(pos, 0, len(days) - 1)]
    nearest = np.minimum(np.abs(target - before), np.abs(target - after))
    return pd.Series(np.clip(nearest, 0, 30).astype("int16"), index=index)


class Baseline:
    """Interface: fit on everything before `origin`, predict the frame given."""

    name = "baseline"

    def fit(self, panel: pd.DataFrame, origin: pd.Timestamp) -> "Baseline":
        self.origin = pd.Timestamp(origin)
        self.past = panel[panel["date"] < self.origin]
        return self

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        raise NotImplementedError


class Zero(Baseline):
    """Predict nothing ever sells. The floor every other number is read against."""

    name = "zero"

    def predict(self, frame):
        return np.zeros(len(frame))


class LastValue(Baseline):
    """Carry the last observed day forward for sixteen days."""

    name = "last_value"

    def fit(self, panel, origin):
        super().fit(panel, origin)
        last_day = self.past["date"].max()
        self.value = (self.past[self.past["date"] == last_day]
                      .set_index(KEYS)["sales"])
        return self

    def predict(self, frame):
        return frame.set_index(KEYS).index.map(self.value).to_numpy(float)


class SeasonalNaive(Baseline):
    """Carry the same weekday from the most recent complete week."""

    name = "seasonal_naive"

    def fit(self, panel, origin):
        super().fit(panel, origin)
        last = self.past["date"].max()
        week = self.past[self.past["date"] > last - pd.Timedelta(days=7)].copy()
        week["dayofweek"] = week["date"].dt.dayofweek
        self.value = week.set_index(KEYS + ["dayofweek"])["sales"]
        return self

    def predict(self, frame):
        idx = frame.assign(dayofweek=frame["date"].dt.dayofweek) \
                   .set_index(KEYS + ["dayofweek"]).index
        return idx.map(self.value).to_numpy(float)


class DowMean(Baseline):
    """Mean of log1p sales on the same weekday over the last `weeks` weeks.

    The one to beat. It is three lines of pandas and it captures the two things
    that carry this dataset — the level of each series and its weekly shape —
    which is most of what there is to capture.
    """

    name = "dow_mean"

    def __init__(self, weeks: int = 8):
        self.weeks = weeks
        self.name = f"dow_mean_{weeks}w"

    def fit(self, panel, origin):
        super().fit(panel, origin)
        last = self.past["date"].max()
        window = self.past[self.past["date"] >
                           last - pd.Timedelta(days=7 * self.weeks)].copy()
        window["dayofweek"] = window["date"].dt.dayofweek
        window["y"] = np.log1p(window["sales"].to_numpy(float))
        self.value = window.groupby(KEYS + ["dayofweek"], observed=True)["y"].mean()
        return self

    def predict(self, frame):
        idx = frame.assign(dayofweek=frame["date"].dt.dayofweek) \
                   .set_index(KEYS + ["dayofweek"]).index
        return np.expm1(np.nan_to_num(idx.map(self.value).to_numpy(float)))


class DirectGBM(Baseline):
    """One gradient booster over every series, trained to forecast directly.

    Direct rather than recursive: a row is (series, origin, horizon), so the
    model learns "sixteen days out" as its own problem instead of feeding its
    own predictions back in sixteen times and compounding the error.

    Global rather than per-series: 1,782 series share one model, so a quiet
    series borrows the weekly shape and holiday response of the 1,781 others.
    Fitting 1,782 separate models on four years of daily data is the obvious
    alternative and it is worse — most of these series do not have enough
    signal of their own to estimate a holiday effect.
    """

    name = "direct_gbm"

    # The column the residual target is measured against: the series' own mean
    # for this weekday over the last eight weeks.
    ANCHOR = "dow_mean_8w"

    def __init__(self, n_origins: int = 52, origin_step: int = 7,
                 horizon: int = 16, seed: int = 0, residual: bool = False,
                 half_life: float | None = None, **kw):
        self.n_origins = n_origins
        self.origin_step = origin_step
        self.horizon = horizon
        # Predict the gap from the weekday mean rather than the level itself.
        # The level is the easy part and it dominates the loss, so a tree spends
        # most of its splits rediscovering that this store sells more bread than
        # that one sells car parts. Subtracting it first spends every split on
        # the part that is hard.
        self.residual = residual
        # Weight an origin by how recent it is. The sixteen days to forecast sit
        # immediately after the training data, so last month's shopping is worth
        # more evidence than last year's.
        self.half_life = half_life
        self.params = dict(
            max_iter=800, learning_rate=0.05, max_leaf_nodes=63,
            min_samples_leaf=40, l2_regularization=1.0,
            early_stopping=False, random_state=seed)
        self.params.update(kw)

    # -- feature assembly ---------------------------------------------------

    def _context(self, panel: pd.DataFrame) -> None:
        # The calendar has to run past the last row the panel holds: in a
        # backtest the panel stops at the fold origin, but the rows to be
        # predicted are the sixteen days after it. Building it from the panel
        # alone leaves those days unmatched and every calendar feature NaN.
        index = pd.date_range(panel["date"].min(),
                              max(panel["date"].max(),
                                  self.origin + pd.Timedelta(days=self.horizon)),
                              freq="D")
        self._oil = oil_series(self._frames.oil, index)
        self._hol = holiday_flags(self._frames.holidays, self._frames.stores, index)
        self._cal = calendar(pd.Series(index)).assign(date=index)
        self._to_holiday = _days_to_national(self._frames.holidays, index)
        self._season = seasonal_index(panel[panel["split"] == "train"])

    def _design(self, panel: pd.DataFrame, origin: pd.Timestamp,
                target: pd.DataFrame) -> pd.DataFrame:
        hist = series_history(panel, origin)
        x = target.merge(hist, on=KEYS, how="left")
        x = x.merge(self._cal, on="date", how="left")
        x = x.merge(self._hol, on=["date", "store_nbr"], how="left")
        x = x.merge(self._season, on=KEYS + ["date"], how="left")
        x["oil"] = x["date"].map(self._oil)
        x["horizon"] = (x["date"] - origin).dt.days.astype("int16") + 1
        x["days_to_holiday"] = x["date"].map(self._to_holiday)
        # A promotion only means something against what this series usually
        # runs: `onpromotion = 4` is a push for one family and a quiet day for
        # another.
        x["promo_vs_hist"] = x["onpromotion"] - x["promo_mean_28"] * 1.0
        # The same-weekday history for *this* row's weekday, picked out of the
        # seven columns the history builder produced.
        for weeks in DOW_WEEKS:
            cols = [f"dow{i}_mean_{weeks}w" for i in range(7)]
            for c in cols:
                if c not in x:
                    x[c] = 0.0
            x[f"dow_mean_{weeks}w"] = x[cols].to_numpy()[
                np.arange(len(x)), x["dayofweek"].to_numpy()]
            x = x.drop(columns=cols)
        return x

    FEATURES_DROP = {"date", "id", "sales", "split", "city", "state", "type"}

    def _matrix(self, x: pd.DataFrame):
        cols = [c for c in x.columns if c not in self.FEATURES_DROP]
        m = x[cols].copy()
        m["family"] = m["family"].map(self._family_code).astype("int16")
        return m[sorted(cols)], sorted(cols)

    # -- fit / predict ------------------------------------------------------

    def fit(self, panel: pd.DataFrame, origin: pd.Timestamp, frames=None):
        self.origin = pd.Timestamp(origin)
        self._frames = frames
        self._family_code = {f: i for i, f in
                             enumerate(sorted(panel["family"].unique()))}
        self._context(panel)

        rows = []
        for k in range(self.n_origins):
            o = self.origin - pd.Timedelta(days=self.origin_step * (k + 1))
            end = o + pd.Timedelta(days=self.horizon - 1)
            target = panel[(panel["date"] >= o) & (panel["date"] <= end) &
                           (panel["split"] == "train")]
            if target.empty:
                continue
            rows.append(self._design(panel, o, target))
        design = pd.concat(rows, ignore_index=True)

        # Dead series are answered by rule, so they are not in the fit: their
        # truth is a constant and they would only pull the loss around.
        alive = ~design.set_index(KEYS).index.isin(
            dead_series(panel, self.origin))
        design = design[alive]

        # Nor are the months before a series was first stocked. Those zeros are
        # a fact about the store's range, not a forecasting problem, and they
        # teach the model to predict zero for a series that is now healthy.
        started = first_sale(panel[panel["split"] == "train"])
        launch = design.set_index(KEYS).index.map(started)
        design = design[design["date"].to_numpy() >=
                        (pd.to_datetime(launch) +
                         pd.Timedelta(days=14)).to_numpy()]

        X, self._cols = self._matrix(design)
        y = np.log1p(design["sales"].to_numpy(float))
        if self.residual:
            y = y - design[self.ANCHOR].to_numpy(float)

        weight = None
        if self.half_life:
            age = (self.origin - design["date"]).dt.days.to_numpy(float)
            weight = 0.5 ** (age / self.half_life)

        cat = [self._cols.index("family"), self._cols.index("store_nbr"),
               self._cols.index("cluster")]
        self.model = HistGradientBoostingRegressor(
            categorical_features=cat, **self.params).fit(X, y, sample_weight=weight)

        self._panel = panel
        self._dead = dead_series(panel, self.origin)
        return self

    def predict(self, frame: pd.DataFrame) -> np.ndarray:
        x = self._design(self._panel, self.origin, frame)
        X, _ = self._matrix(x)
        pred_log = self.model.predict(X[self._cols])
        if self.residual:
            pred_log = pred_log + x[self.ANCHOR].to_numpy(float)
        out = np.expm1(pred_log)
        dead = x.set_index(KEYS).index.isin(self._dead)
        out[dead] = 0.0
        return np.clip(out, 0.0, None)


class Blend(Baseline):
    """Average several variants in log space.

    Not an ensemble for its own sake. Every variant tried here lands within
    0.003 of every other on the aggregate while disagreeing on which rows it
    gets wrong — the residual target wins the June fold and loses the August
    one, the lag features do the reverse. Averaging keeps the agreement and
    cancels part of the disagreement, which is the one reliable gain left once
    a feature search has stopped paying.

    Averaged in log space because that is the space the metric lives in;
    averaging the sales and taking the log afterwards is a different, worse
    estimator on a series that is zero a third of the time.
    """

    name = "blend"

    def __init__(self, variants: list[dict] | None = None):
        self.variants = variants or [dict(), dict(residual=True),
                                     dict(half_life=180)]
        self.name = f"blend_{len(self.variants)}"

    def fit(self, panel, origin, frames=None):
        self.origin = pd.Timestamp(origin)
        self.models = [DirectGBM(**v).fit(panel, origin, frames=frames)
                       for v in self.variants]
        return self

    def predict(self, frame):
        logs = [np.log1p(np.clip(m.predict(frame), 0.0, None))
                for m in self.models]
        return np.expm1(np.mean(logs, axis=0))

### Backtest

In [ ]:
def run(builders, n_folds=3):
    rows = []
    for start, end in make_folds(train["date"].max(), n=n_folds):
        holdout = train[(train.date >= start) & (train.date <= end)]
        history = pd.concat([panel[panel.date < start], test])
        absent = dead_series(panel, start)
        for build in builders:
            model, t0 = build(), time.perf_counter()
            learned = isinstance(model, (DirectGBM, Blend))
            model.fit(history, start, **({"frames": frames} if learned else {}))
            pred = np.nan_to_num(model.predict(holdout))
            if not learned:
                # Every model gets the free win, so the table compares the
                # modelling and not who remembered the rule.
                is_dead = holdout.set_index(["store_nbr", "family"]).index.isin(absent)
                pred[is_dead] = 0.0
            rows.append({"fold": f"{start:%m-%d}", "model": model.name,
                         "rmsle": rmsle(holdout.sales, pred),
                         "sec": round(time.perf_counter() - t0)})
            print(f"  {start:%m-%d}  {model.name:16s} {rows[-1]['rmsle']:.5f}"
                  f"  ({rows[-1]['sec']}s)", flush=True)
    return pd.DataFrame(rows)


scores = run([Zero, LastValue, SeasonalNaive,
              lambda: DowMean(4), lambda: DowMean(8), lambda: DowMean(16),
              DirectGBM])
(scores.pivot_table(index="model", columns="fold", values="rmsle")
       .assign(mean=lambda t: t.mean(axis=1))
       .sort_values("mean").round(5))

The weekday mean is the number to beat, and it is three lines of pandas: it
captures the level of each series and its weekly shape, which is most of what
there is to capture here. Everything after that is worth about a tenth.

### Where the loss actually is

RMSLE is a mean over 28,512 rows and it hides everything. Adding features until
the aggregate moves is guessing; splitting the same number says where to aim.

In [ ]:
start, end = make_folds(train.date.max(), n=3)[-1]
holdout = train[(train.date >= start) & (train.date <= end)].copy()
history = pd.concat([panel[panel.date < start], test])

model = DirectGBM().fit(history, start, frames=frames)
holdout["pred"] = np.nan_to_num(model.predict(holdout))
holdout["err2"] = (np.log1p(holdout.pred.clip(lower=0))
                   - np.log1p(holdout.sales)) ** 2
total = holdout.err2.sum()

(holdout.groupby("family")
        .agg(rmsle=("err2", lambda s: float(np.sqrt(s.mean()))),
             share_pct=("err2", lambda s: 100 * s.sum() / total),
             mean_sales=("sales", "mean"))
        .sort_values("share_pct", ascending=False)
        .head(8).round(3))

Five families out of thirty-three carry about 40% of the squared error on 15% of
the rows, and the worst is the one with the sharpest annual season: Ecuadorean
term starts in August and school supplies go up several-fold for a few weeks.
The year-ago seasonal index in the features above was built for exactly that and
moved the August fold from 0.42002 to 0.41305 — real, and small.

Error also climbs across the horizon, from about 0.40 on a fold's first days to
0.46 on its last. That matters for what comes next.

In [ ]:
(holdout.groupby("date").err2
        .apply(lambda s: float(np.sqrt(s.mean())))
        .round(4).to_frame("rmsle").T)

## The result I did not expect

Two submissions — a plain gradient booster, and a three-way blend with
cross-sectional features, explicit lags, a residual target and recency-weighted
origins — differ by **0.021** on the third fold, the sixteen days immediately
before the test window. On the public leaderboard they scored **0.40713** and
**0.40695**. Two hundredths of a percent apart.

The local improvement is real and measured. It is simply not reaching the
leaderboard, and until that is understood, more feature work is guessing.

The hypothesis worth testing first is in the per-day table above: error climbs
steadily across the horizon, so if the public leaderboard is scored on the
earlier part of the test window, it is measuring exactly the stretch where every
one of these models agrees. The test is cheap — score a fold on its first eight
days and its last eight separately, and see which half tracks the leaderboard.

If your own local gains do transfer to the leaderboard, I would genuinely like
to know what you are doing differently.

## Submission

In [ ]:
origin = test.date.min()
final = Blend().fit(panel, origin, frames=frames)

out = test.copy()
out["sales"] = np.clip(np.nan_to_num(final.predict(out)), 0.0, None)
sub = out[["id", "sales"]].astype({"id": int}).sort_values("id", ignore_index=True)

assert len(sub) == 28512, len(sub)
assert sub.sales.notna().all() and (sub.sales >= 0).all()
sub.to_csv("submission.csv", index=False)

print(f"{len(sub)} rows, {(sub.sales == 0).mean():.1%} zeros, "
      f"mean {sub.sales.mean():.1f}")
sub.head()